# AquaVision — Huấn luyện mô hình phát hiện cá bằng RF-DETR (Roboflow)

Notebook này huấn luyện một mô hình **object detection** dạng transformer, họ **RF-DETR** (Roboflow),
trên Google Colab, sử dụng dữ liệu ảnh cá (Koi / Tilapia / Eel) được quản lý và gán nhãn trong dự án
AquaVision, tải trực tiếp từ Roboflow ở định dạng **COCO**.

**Quy trình:**

1. Kiểm tra GPU và runtime
2. Gắn Google Drive để lưu trữ dữ liệu & kết quả lâu dài
3. Cài đặt thư viện
4. Cấu hình huấn luyện — **một cell duy nhất**, chỉnh sửa ở đây, các cell phía dưới không hard-code lại
5. Xác thực & tải dataset (định dạng COCO) từ Roboflow, kiểm tra tính toàn vẹn
6. Thiết lập reproducibility (seed)
7. Huấn luyện mô hình (hỗ trợ resume qua checkpoint PyTorch Lightning)
8. Đánh giá mô hình trên tập test
9. Trực quan hoá kết quả (đường log loss/mAP qua TensorBoard)
10. Suy luận thử nghiệm nhanh (vẽ bounding box bằng `supervision`)
11. Export mô hình (ONNX — tuỳ chọn)
12. Ghi manifest (thông số + kết quả) về Google Drive để truy vết
13. Tổng kết

> **Trước khi chạy:** vào `Runtime > Change runtime type` và chọn GPU (T4 trở lên được khuyến nghị).
>
> **Cần chuẩn bị:** một Roboflow API key. Khuyến nghị lưu vào Colab Secrets (biểu tượng 🔑 ở thanh bên
> trái) với tên `ROBOFLOW_API_KEY` thay vì dán trực tiếp vào notebook.
>
> **Khác biệt so với YOLO/RT-DETR:** RF-DETR đọc dataset ở định dạng **COCO** — mỗi thư mục split
> (`train/`, `valid/`, `test/`) chứa ảnh cùng một file `_annotations.coco.json`, thay vì file `.txt`
> theo từng ảnh như YOLO. Vì vậy dataset phải được tải từ Roboflow với format `"coco"`, **không** dùng
> lại dataset đã tải cho hai notebook YOLO/RT-DETR.

## 1. Kiểm tra môi trường GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Không tìm thấy GPU. Vào Runtime > Change runtime type > Hardware accelerator "
    "và chọn GPU (khuyến nghị T4 trở lên) trước khi tiếp tục."
)

print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"CUDA    : {torch.version.cuda}")
print(f"PyTorch : {torch.__version__}")

## 2. Gắn Google Drive

Dataset, checkpoint và toàn bộ kết quả huấn luyện được ghi trực tiếp vào Google Drive để:

- Không mất dữ liệu khi Colab tự ngắt phiên (idle timeout / disconnect)
- Có thể **resume** huấn luyện từ checkpoint gần nhất mà không cần tải lại dataset hoặc bắt đầu lại từ đầu

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Cài đặt thư viện

In [ ]:
%pip install -q "rfdetr[train]" "roboflow>=1.1.0" "supervision>=0.25.0"

## 4. Cấu hình huấn luyện

Đây là **cell cấu hình duy nhất** của notebook. Điều chỉnh dataset Roboflow, kích thước model và
hyperparameter tại đây.

- `model.size` hỗ trợ: `n` (Nano), `s` (Small), `m` (Medium), `l` (Large)
- `hyperparameters.resolution`: để `None` để dùng resolution mặc định (đã tối ưu cho pretrained
  weights) của từng biến thể model; chỉ override khi biết rõ đánh đổi độ chính xác/tốc độ
- `hyperparameters.resume_checkpoint`: `None` để huấn luyện mới, `"last"` để tiếp tục từ checkpoint
  gần nhất trong `output_dir`, hoặc đường dẫn `.ckpt` cụ thể

In [ ]:
import json

CONFIG = {
    "project": {
        # Dùng để đặt tên thư mục run trên Drive — nên gồm loài cá + kiến trúc model
        "name": "aquavision-koi-rfdetr",
    },
    "roboflow": {
        "workspace": "dd-1pubd",
        "project": "koi-f2dp4",
        "version": 2,
        "format": "coco",  # RF-DETR yêu cầu định dạng COCO (train/valid/test + _annotations.coco.json)
    },
    "model": {
        "size": "m",  # n | s | m | l
    },
    "hyperparameters": {
        "epochs": 100,
        "batch_size": 8,
        "grad_accum_steps": 4,   # batch hiệu dụng = batch_size * grad_accum_steps
        "lr": 1e-4,
        "resolution": None,      # None = dùng resolution mặc định của biến thể model
        "early_stopping": True,
        "early_stopping_patience": 10,
        "seed": 42,
        # None = huấn luyện mới; "last" = tiếp tục checkpoint gần nhất; hoặc đường dẫn .ckpt cụ thể
        "resume_checkpoint": None,
    },
    "export": {
        "onnx": False,
    },
    "drive": {
        # Toàn bộ dataset/checkpoint/kết quả được ghi vào đây để không mất khi mất phiên Colab
        "root": "/content/drive/MyDrive/AquaVision",
    },
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False, default=str))

## 5. Xác thực & tải dataset từ Roboflow

API key được lấy an toàn qua Colab Secrets dưới tên `ROBOFLOW_API_KEY`. Nếu chưa cấu hình secret,
notebook sẽ yêu cầu nhập trực tiếp (giá trị chỉ tồn tại trong runtime hiện tại, không được lưu vào file).

In [ ]:
import os
from getpass import getpass

ROBOFLOW_API_KEY = None

try:
    from google.colab import userdata

    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Nhập Roboflow API key (không được lưu lại trong notebook): ")

os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

In [ ]:
from pathlib import Path

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_project = rf.workspace(CONFIG["roboflow"]["workspace"]).project(CONFIG["roboflow"]["project"])
rf_version = rf_project.version(CONFIG["roboflow"]["version"])

dataset_dir = Path("/content/datasets") / CONFIG["project"]["name"]
dataset = rf_version.download(CONFIG["roboflow"]["format"], location=str(dataset_dir))

print(f"Dataset đã tải về: {dataset.location}")

In [ ]:
import json as _json

train_annotations_path = Path(dataset.location) / "train" / "_annotations.coco.json"
assert train_annotations_path.exists(), (
    f"Không tìm thấy {train_annotations_path}. RF-DETR yêu cầu định dạng COCO — kiểm tra lại "
    f"CONFIG['roboflow']['format'] có đúng là 'coco' không."
)

class_names = None

for split in ("train", "valid", "test"):
    annotations_path = Path(dataset.location) / split / "_annotations.coco.json"
    if not annotations_path.exists():
        print(f"{split:>6}: (không có _annotations.coco.json — bỏ qua)")
        continue

    with open(annotations_path) as f:
        coco = _json.load(f)

    print(f"{split:>6}: {len(coco['images'])} ảnh, {len(coco['annotations'])} annotation")

    if split == "train":
        class_names = [c["name"] for c in sorted(coco["categories"], key=lambda c: c["id"])]

print(f"Tên lớp: {class_names}")

## 6. Thiết lập reproducibility

Cố định seed cho toàn bộ nguồn ngẫu nhiên để kết quả có thể tái lập giữa các lần chạy.

In [ ]:
import random

import numpy as np


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["hyperparameters"]["seed"])

## 7. Huấn luyện mô hình

RF-DETR fine-tune từ checkpoint pretrained COCO tương ứng với `model.size`. Kết quả được ghi trực tiếp
vào Google Drive (`output_dir`).

Nếu phiên Colab bị ngắt giữa chừng: đặt `CONFIG["hyperparameters"]["resume_checkpoint"] = "last"` ở
Bước 4 rồi chạy lại notebook từ đầu — PyTorch Lightning sẽ tự đọc checkpoint gần nhất trong `output_dir`
và tiếp tục đúng từ epoch bị ngắt (yêu cầu `output_dir` phải là cùng một thư mục trên Drive giữa hai
lần chạy, mặc định notebook đã đảm bảo điều này).

In [ ]:
from rfdetr import RFDETRLarge, RFDETRMedium, RFDETRNano, RFDETRSmall

_RFDETR_MODEL_CLASSES = {
    "n": RFDETRNano,
    "s": RFDETRSmall,
    "m": RFDETRMedium,
    "l": RFDETRLarge,
}


def resolve_rfdetr_class(size: str):
    model_cls = _RFDETR_MODEL_CLASSES.get(size)
    if model_cls is None:
        supported = ", ".join(_RFDETR_MODEL_CLASSES)
        raise ValueError(f"Size '{size}' không được hỗ trợ (hỗ trợ: {supported})")
    return model_cls

In [ ]:
model_cfg = CONFIG["model"]
hyperparams = CONFIG["hyperparameters"]

model_cls = resolve_rfdetr_class(model_cfg["size"])
print(f"Model: {model_cls.__name__}")

output_dir = Path(CONFIG["drive"]["root"]) / "runs" / "rfdetr" / CONFIG["project"]["name"]
output_dir.mkdir(parents=True, exist_ok=True)

model = model_cls()

train_kwargs = dict(
    dataset_dir=str(dataset.location),
    epochs=hyperparams["epochs"],
    batch_size=hyperparams["batch_size"],
    grad_accum_steps=hyperparams["grad_accum_steps"],
    lr=hyperparams["lr"],
    early_stopping=hyperparams["early_stopping"],
    early_stopping_patience=hyperparams["early_stopping_patience"],
    seed=hyperparams["seed"],
    output_dir=str(output_dir),
    class_names=class_names,
    run_test=True,
)

if hyperparams["resolution"] is not None:
    train_kwargs["resolution"] = hyperparams["resolution"]

if hyperparams["resume_checkpoint"] is not None:
    train_kwargs["resume"] = hyperparams["resume_checkpoint"]
    print(f"Tiếp tục huấn luyện từ checkpoint: {hyperparams['resume_checkpoint']}")

model.train(**train_kwargs)

best_weights = output_dir / "checkpoint_best_total.pth"
last_weights = output_dir / "checkpoint_best_regular.pth"

print(f"Best weights: {best_weights}")
print(f"Last weights: {last_weights}")

## 8. Đánh giá mô hình

`run_test=True` ở Bước 7 đã tự động đánh giá trên tập `test` sau khi huấn luyện kết thúc — kết quả
COCO mAP được in ra trong log huấn luyện phía trên. Nạp lại checkpoint tốt nhất để xác nhận và dùng
cho các bước suy luận tiếp theo.

In [ ]:
best_model = model_cls.from_checkpoint(str(best_weights))

print(f"Đã nạp checkpoint: {best_weights}")

## 9. Trực quan hoá kết quả

RF-DETR ghi log qua TensorBoard (loss, mAP theo epoch) vào `output_dir/tensorboard/`. Mở trực tiếp
trong notebook bằng magic command dưới đây.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {output_dir}

## 10. Suy luận thử nghiệm

In [ ]:
import supervision as sv
from IPython.display import display
from PIL import Image as PILImage

test_images_dir = Path(dataset.location) / "test"

if test_images_dir.exists():
    sample_images = sorted(
        p for p in test_images_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )[:5]

    for image_path in sample_images:
        image = PILImage.open(image_path)
        detections = best_model.predict(image, threshold=0.5)

        labels = [
            f"{class_names[class_id]} {confidence:.2f}"
            for class_id, confidence in zip(detections.class_id, detections.confidence)
        ]

        annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
        annotated = sv.LabelAnnotator().annotate(annotated, detections, labels)

        display(annotated)
else:
    print("Không có thư mục test/ — bỏ qua suy luận thử nghiệm.")

## 11. Export mô hình (tuỳ chọn)

Bật `CONFIG["export"]["onnx"]` ở Bước 4 nếu cần triển khai lên edge device.

In [ ]:
export_cfg = CONFIG["export"]

if export_cfg["onnx"]:
    onnx_path = best_model.export(output_dir=str(output_dir), format="onnx")
    print(f"ONNX export: {onnx_path}")

## 12. Ghi manifest kết quả

Lưu lại cấu hình của lần chạy này để truy vết sau này (dataset version, hyperparameter, checkpoint
dùng). Metrics chi tiết (COCO mAP) đã được ghi trong log huấn luyện và TensorBoard ở Bước 9.

In [ ]:
import json
from datetime import datetime

run_manifest = {
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "config": CONFIG,
    "model_class": model_cls.__name__,
    "best_weights": str(best_weights),
    "last_weights": str(last_weights),
    "class_names": class_names,
}

manifest_path = output_dir / "run_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(run_manifest, f, indent=2, default=str, ensure_ascii=False)

print(f"Manifest: {manifest_path}")

## 13. Tổng kết

In [ ]:
print("Huấn luyện hoàn tất")
print(f"  Model         : {model_cls.__name__}")
print(f"  Best weights  : {best_weights}")
print(f"  Kết quả lưu tại: {output_dir}")
print("  Xem chi tiết mAP/loss trong log huấn luyện ở Bước 7 hoặc TensorBoard ở Bước 9.")